# 01 · Limpeza dos preços ANP (RJ)

Objetivo: transformar os CSVs mensais nacionais da ANP (2023–2025) em uma base limpa do estado do Rio de Janeiro, com cada descarte registrado.

Fonte e licença: ver [docs/fontes.md](../docs/fontes.md). Licença confirmada pelo dono do projeto.

Pergunta que esta etapa sustenta: [README.md](../README.md).

Cada passo grava uma linha na tabela de decisões. A tabela final é salva em `docs/decisoes_limpeza.md`.

In [1]:
from pathlib import Path
import glob

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = RAIZ / "data" / "raw"
PROCESSADO = RAIZ / "data" / "processed"
DOCS = RAIZ / "docs"
PROCESSADO.mkdir(parents=True, exist_ok=True)

decisoes = []


def registra(etapa, entrada, saida, motivo):
    decisoes.append({"etapa": etapa, "linhas_entrada": entrada, "linhas_saida": saida,
                     "descartadas": entrada - saida, "motivo": motivo})

## 1. Carga

Os CSVs usam `;` como separador, BOM UTF-8 e vírgula decimal. Todas as colunas são lidas como texto para controlar a conversão.

In [2]:
arquivos = sorted(glob.glob(str(RAW / "*.csv")))
partes = [
    pd.read_csv(f, sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False).assign(arquivo=Path(f).name)
    for f in arquivos
]
df = pd.concat(partes, ignore_index=True)
print(f"{len(arquivos)} arquivos, {len(df):,} linhas no total")
df.groupby("arquivo").size().head()

72 arquivos, 2,608,591 linhas no total


arquivo
diesel-gnv-2023-01.csv    23438
diesel-gnv-2023-02.csv    20036
diesel-gnv-2023-03.csv    23308
diesel-gnv-2023-04.csv    22196
diesel-gnv-2023-05.csv    28087
dtype: int64

## 2. Recorte: estado do Rio de Janeiro

Filtro pela sigla da UF. O recorte é o definido no `PROMPT.md`; mudar exige confirmação do dono do projeto.

In [3]:
n0 = len(df)
rj = df[df["Estado - Sigla"].str.strip() == "RJ"].copy()
registra("UF = RJ", n0, len(rj), "Recorte definido no PROMPT.md")
print(f"{len(rj):,} linhas do RJ")

214,065 linhas do RJ


## 3. Produtos

Mantidos: gasolina comum (`GASOLINA`), etanol hidratado (`ETANOL`), óleo diesel (`DIESEL`, `DIESEL S10`).

Descartados:
- `GASOLINA ADITIVADA`: outro produto, com preço próprio. Misturar inflaria a série de gasolina comum.
- `GNV`: medido em R$/m³, não em R$/litro. Não é comparável com etanol.

In [4]:
n1 = len(rj)
PRODUTOS = ["GASOLINA", "ETANOL", "DIESEL", "DIESEL S10"]
rj = rj[rj["Produto"].str.strip().isin(PRODUTOS)].copy()
registra("Produto em {gasolina, etanol, diesel}", n1, len(rj),
         "Aditivada tem preço próprio; GNV é em R$/m³")
print(df.loc[df["Estado - Sigla"].str.strip() == "RJ", "Produto"].value_counts())

Produto
GASOLINA              50310
ETANOL                46510
GASOLINA ADITIVADA    45115
DIESEL S10            31645
GNV                   21761
DIESEL                18724
Name: count, dtype: int64


## 4. Preço: vírgula decimal e valores sem vírgula

O preço vem como `5,99`. A conversão troca `,` por `.` e força número.

Valores com uma casa decimal (ex.: `6,2`) são aceitos como `6,20`: a ANP parece omitir zeros à direita. Essa é uma inferência a partir do padrão dos dados, não uma regra documentada.

Há linhas sem vírgula decimal (ex.: `6`). Não dá para saber se é R$ 6,00 ou outro valor. Essas linhas são descartadas, não convertidas.

In [5]:
valor = rj["Valor de Venda"].str.strip()
padrao = valor.str.match(r"^\d+,\d{1,2}$")
sem_padrao = rj.loc[~padrao, "Valor de Venda"].value_counts()
print("valores fora do padrão 'N,NN':")
print(sem_padrao)

n2 = len(rj)
rj = rj[padrao].copy()
rj["preco"] = rj["Valor de Venda"].str.replace(",", ".", regex=False).astype(float)
registra("Valor sem vírgula decimal", n2, len(rj),
         "Valor como '5' é ambíguo (R$ 5,00 ou truncado); descartado")

valores fora do padrão 'N,NN':
Valor de Venda
6    40
5     3
4     2
Name: count, dtype: int64


## 5. Data e município

Data da coleta em `dd/mm/aaaa` vira `datetime`. A semana usada nas análises é a semana ISO da coleta.

Município: remove espaços nas bordas. Os nomes já vêm sem acento e em maiúsculas, o que foi verificado antes da escrita deste notebook.

In [6]:
n3 = len(rj)
rj["data"] = pd.to_datetime(rj["Data da Coleta"].str.strip(), format="%d/%m/%Y", errors="coerce")
rj = rj.dropna(subset=["data"]).copy()
registra("Data inválida", n3, len(rj), "Data não parseável em dd/mm/aaaa")

rj["municipio"] = rj["Municipio"].str.strip()
rj["semana"] = rj["data"].dt.isocalendar().week.astype(int)
rj["ano_iso"] = rj["data"].dt.isocalendar().year.astype(int)
print(rj["municipio"].nunique(), "municípios")
print(sorted(rj["municipio"].unique()))

32 municípios
['ANGRA DOS REIS', 'ARARUAMA', 'BARRA DO PIRAI', 'BARRA MANSA', 'BELFORD ROXO', 'CABO FRIO', 'CAMPOS DOS GOYTACAZES', 'DUQUE DE CAXIAS', 'ITABORAI', 'ITAGUAI', 'ITAPERUNA', 'MACAE', 'MAGE', 'MARICA', 'NILOPOLIS', 'NITEROI', 'NOVA FRIBURGO', 'NOVA IGUACU', 'PETROPOLIS', 'RESENDE', 'RIO BONITO', 'RIO DE JANEIRO', 'SANTO ANTONIO DE PADUA', 'SAO FRANCISCO DE ITABAPOANA', 'SAO GONCALO', 'SAO JOAO DE MERITI', 'SAPUCAIA', 'SAQUAREMA', 'TERESOPOLIS', 'TRES RIOS', 'VALENCA', 'VOLTA REDONDA']


## 6. Duplicatas

Linhas idênticas em todas as colunas (exceto o arquivo de origem) são duplicata de carga. Mantém a primeira.

In [7]:
n4 = len(rj)
chave = [c for c in rj.columns if c not in ("arquivo",)]
rj = rj.drop_duplicates(subset=chave, keep="first").copy()
registra("Duplicata exata", n4, len(rj), "Mesma revenda, produto, data e preço em linhas repetidas")

## 7. Outliers

Regra de domínio: preço de combustível por litro entre R$ 2,00 e R$ 12,00. Fora disso é erro de digitação ou unidade.

O limite é largo de propósito. Outlier estatístico (ex.: 3 desvios) poderia cortar preço real em cidade cara ou em semana de reajuste.

In [8]:
n5 = len(rj)
fora = rj[(rj["preco"] < 2) | (rj["preco"] > 12)]
print(f"{len(fora)} linhas fora de R$ 2 a 12")
rj = rj[(rj["preco"] >= 2) & (rj["preco"] <= 12)].copy()
registra("Outlier de domínio", n5, len(rj), "Preço fora de R$ 2,00 a R$ 12,00 por litro")
print(rj["preco"].describe())

0 linhas fora de R$ 2 a 12
count    147142.000000
mean          5.455788
std           0.837073
min           3.300000
25%           4.690000
50%           5.690000
75%           6.090000
max           8.890000
Name: preco, dtype: float64


## 8. Saída

Base final salva em `data/processed/precos_rj.csv`. A pasta `data/` não vai para o Git: o notebook recria o arquivo a partir de `data/raw`.

In [9]:
colunas = {
    "Revenda": "revenda",
    "CNPJ da Revenda": "cnpj_revenda",
    "Bandeira": "bandeira",
    "Produto": "produto",
    "Unidade de Medida": "unidade",
}
saida = rj.rename(columns=colunas)[["municipio", "produto", "data", "semana", "ano_iso",
                                    "preco", "revenda", "cnpj_revenda", "bandeira", "unidade"]]
saida = saida.sort_values(["municipio", "produto", "data"]).reset_index(drop=True)
saida.to_csv(PROCESSADO / "precos_rj.csv", index=False, encoding="utf-8")
print(f"{len(saida):,} linhas salvas")
saida.head()

147,142 linhas salvas


,municipio,produto,data,semana,ano_iso,preco,revenda,cnpj_revenda,bandeira,unidade
0,ANGRA DOS REIS,DIESEL,2023-01-02,1,2023,7.64,POSTO PRAIA DA RIBEIRA LTDA,03.825.735/0001-42,IPIRANGA,R$ / litro
1,ANGRA DOS REIS,DIESEL,2023-01-03,1,2023,7.29,POSTO DOS SANTOS REIS LTDA.,29.051.570/0001-92,VIBRA ENERGIA,R$ / litro
2,ANGRA DOS REIS,DIESEL,2023-01-03,1,2023,6.39,POSTO JACUECANGA LTDA.,17.823.690/0001-81,BRANCA,R$ / litro
3,ANGRA DOS REIS,DIESEL,2023-01-09,2,2023,6.39,POSTO JACUECANGA LTDA.,17.823.690/0001-81,BRANCA,R$ / litro
4,ANGRA DOS REIS,DIESEL,2023-01-11,2,2023,6.29,S.J. POSTO DE COMBUSTÍVEIS LTDA.,14.637.692/0001-98,ALESAT,R$ / litro


## 9. Tabela de decisões

Tabela com cada descarte. É copiada para `docs/decisoes_limpeza.md`.

In [10]:
tabela = pd.DataFrame(decisoes)
print(tabela.to_string(index=False))
linhas = ["| Etapa | Entrada | Saída | Descartadas | Motivo |", "|---|---:|---:|---:|---|"]
for r in tabela.itertuples():
    linhas.append(f"| {r.etapa} | {r.linhas_entrada:,} | {r.linhas_saida:,} | {r.descartadas:,} | {r.motivo} |")
(DOCS / "decisoes_limpeza.md").write_text(
    "# Decisões de limpeza\n\nGerado por notebooks/01_limpeza.ipynb.\n\n" + "\n".join(linhas) + "\n",
    encoding="utf-8")
print("gravado", DOCS / "decisoes_limpeza.md")

                                etapa  linhas_entrada  linhas_saida  descartadas                                                     motivo
                              UF = RJ         2608591        214065      2394526                              Recorte definido no PROMPT.md
Produto em {gasolina, etanol, diesel}          214065        147189        66876                Aditivada tem preço próprio; GNV é em R$/m³
            Valor sem vírgula decimal          147189        147144           45 Valor como '5' é ambíguo (R$ 5,00 ou truncado); descartado
                        Data inválida          147144        147144            0                           Data não parseável em dd/mm/aaaa
                      Duplicata exata          147144        147142            2   Mesma revenda, produto, data e preço em linhas repetidas
                   Outlier de domínio          147142        147142            0                 Preço fora de R$ 2,00 a R$ 12,00 por litro
gravado E:\Programac